## RAG Fusion Pipeline - Ensemble Retriever

This notebook demonstrates RAG Fusion using **two retrieval strategies combined via an Ensemble Retriever**. Instead of generating sub-queries with an LLM, we use two different retrieval strategies on the same vector store and fuse their results using **Reciprocal Rank Fusion (RRF)** built into LangChain's `EnsembleRetriever`.

**Retrievers used:**

- **Similarity Search** - retrieves documents based on cosine similarity.
- **MMR (Maximal Marginal Relevance)** - balances relevance with diversity (`lambda_mult=0.5`).

**Steps covered:**

1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store them in ChromaDB
4. Create two retrievers (Similarity Search + MMR)
5. Apply RAG Fusion using an Ensemble Retriever with weights `[0.5, 0.5]`
6. Fuse the retrieved results using Reciprocal Rank Fusion (RRF)
7. **Augmentation** - build context from the fused documents
8. **Generation** - produce a grounded answer using an LLM

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from rag_fusion import RAGFusion

/tmp/ipykernel_1247143/2620883329.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


In [4]:
llm = ChatOllama(
    model="smollm2:360m",
    temperature=0

) 
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


In [6]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="notebooklm_rag_ensemble"
)

print("Vector store created successfully.")


Vector store created successfully.


In [7]:
# Retriever 1: standard cosine similarity search
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
# Retriever 2: MMR - promotes diverse results alongside relevant ones
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

In [ ]:
# RAGFusion.from_retrievers wraps both retrievers inside LangChain's EnsembleRetriever. 
# Equal weights [0.5, 0.5] mean both retrievers contribute equally to the final RRF ranking. 
# When invoke is called, the ensemble retriever queries both strategies and merges the results.
rag_fusion = RAGFusion.from_retrievers(
    base_retrievers=[similarity_retriever, mmr_retriever],
    weights=[0.5, 0.5],
    k=3
)

In [9]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# Both retrievers run in parallel; EnsembleRetriever fuses the results via RRF
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
identifies which specific passages from the uploaded documents supported each claim in the answer. These
citations are surfaced to the user as inline references, allowing them to verify the accuracy of the response by
reading the original source material.
NotebookLM also handles cases where the retrieved context is insufficient to answer the question. In such
cases, the system is designed to explicitly acknowledge the limitation and inform the user that the answer

--- Document 2 ---
cannot be found in the uploaded sources, rather than fabricating information. This behavior is a direct result
of the grounding mechanism that constrains generation to the retrieved evidence.
6. Conclusion
NotebookLM represents a practical and user-friendly implementation of Retrieval-Augmented Generation
technology. By combining intelligent document processing, semantic vector search, and grounded language

--- Document 3 ---
material provided by the user

In [10]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

identifies which specific passages from the uploaded documents supported each claim in the answer. These
citations are surfaced to the user as inline references, allowing them to verify the accuracy of the response by
reading the original source material.
NotebookLM also handles cases where the retrieved context is insufficient to answer the question. In such
cases, the system is designed to explicitly acknowledge the limitation and inform the user that the answer

cannot be found in the uploaded sources, rather than fabricating information. This behavior is a direct result
of the grounding mechanism that constrains generation to the retrieved evidence.
6. Conclusion
NotebookLM represents a practical and user-friendly implementation of Retrieval-Augmented Generation
technology. By combining intelligent document processing, semantic vector search, and grounded language

material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need

In [12]:

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

NotebookLM retrieves relevant information from uploaded documents by using semantic vector search and grounded language material provided by the user. This ensures that the generated answer is based on the most accurate and relevant information available from the original sources.


--- Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.

--- Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately

--- Compressed Result 1 [medicine] ---
*Personalized medicine*

The use of *CRISPR* gene editing technology in personalized medicine has been a game-changer. Researchers are leveraging *genomic data* to develop personalized treatment approaches, tailoring medications to an individual's unique genetic profile.

--- Compressed Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately modeling protein folding and chemical interactions. Major tech companies and governments are investing billions in quantum research, viewing it as essential for national security and economic competitiveness.

--- Compressed Result 3 [artificial_intelligence] ---
* The AI has made rema

--- Pipeline Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.



--- Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.

--- Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately

--- Compressed Result 1 [medicine] ---
*Personalized medicine*

The use of *CRISPR* gene editing technology in personalized medicine has been a game-changer. Researchers are leveraging *genomic data* to develop personalized treatment approaches, tailoring medications to an individual's unique genetic profile.

--- Compressed Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately modeling protein folding and chemical interactions. Major tech companies and governments are investing billions in quantum research, viewing it as essential for national security and economic competitiveness.

--- Compressed Result 3 [artificial_intelligence] ---
* The AI has made rema

--- Pipeline Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.



--- Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.

--- Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately

--- Compressed Result 1 [medicine] ---
*Personalized medicine*

The use of *CRISPR* gene editing technology in personalized medicine has been a game-changer. Researchers are leveraging *genomic data* to develop personalized treatment approaches, tailoring medications to an individual's unique genetic profile.

--- Compressed Result 2 [quantum_computing] ---
Quantum computing has reached a critical milestone with several companies demonstrating quantum advantage on specific computational tasks. Error correction remains the biggest challenge, as current quantum processors are highly susceptible to noise and decoherence. Quantum simulation of molecular structures could transform drug discovery by accurately modeling protein folding and chemical interactions. Major tech companies and governments are investing billions in quantum research, viewing it as essential for national security and economic competitiveness.

--- Compressed Result 3 [artificial_intelligence] ---
* The AI has made rema

--- Pipeline Result 1 [medicine] ---
CRISPR gene editing technology has revolutionized medical genomics, enabling precise modifications to DNA sequences that were previously impossible. Researchers are using genomic data to develop personalized medicine approaches, tailoring treatments based on an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare genetic disorders. Hospital information systems are increasingly integrating genomic data to support clinical decision-making at the point of care.

